# Weighted soft voting over the three backbones

Reads the class probabilities of the three models and combines them with weights
0.3 / 0.4 / 0.3 for ResNet50 / InceptionV3 / EfficientNetB0, as in Section 3.6.

**One caveat, stated plainly.** The probability file of the ResNet50 run reported in the
paper no longer exists: it was overwritten on 24 July 2025 by a later training run, and the
surviving weights do not reproduce it (91.4% per-image agreement). This notebook therefore
uses `resnet_pred_regenerated.csv`, a re-inference from those weights, which scores 87.89%
instead of 89.62%. Run end to end it gives an ensemble accuracy of about **92.27%**, against
the **92.35%** reported.

The reported numbers themselves are in `predictions/combined_model_predictions.csv`, which
holds the per-image predictions of all three models and of the ensemble from the reported
run. `reproduce_results.py` recomputes every figure in Tables 2, 4, 5 and 7 from it exactly.

That the ensemble lands within 0.08 points despite a ResNet run 1.7 points weaker is itself
evidence that the reported result does not depend on one lucky file.

Run this notebook from the `notebooks/` folder.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score,accuracy_score,classification_report

In [ ]:
gt = pd.read_csv("../predictions/test_ground_truth.csv")
y_test = gt["actual"]


In [ ]:
y_test

In [ ]:
# NOTE: the probabilities of the ResNet50 run reported in the paper were lost to an
# overwrite; this is a re-inference from the surviving weights (87.89% vs 89.62%).
resnet_df = pd.read_csv("../predictions/resnet_pred_regenerated.csv")
resnet_df.head()


In [ ]:
resnet_pred = np.argmax(resnet_df, axis=1)
resnet_pred

In [ ]:
resnet_accuracy = accuracy_score(y_test, resnet_pred)
resnet_accuracy

In [ ]:
print(classification_report(y_test,resnet_pred,digits=4))

In [ ]:
inception_df = pd.read_csv("../predictions/inception_pred.csv")
inception_df.head()


In [ ]:
inception_pred = np.argmax(inception_df, axis=1)
inception_pred

In [ ]:
inception_accuracy = accuracy_score(y_test, inception_pred)
inception_accuracy

In [ ]:
print(classification_report(y_test,inception_pred,digits=4))

In [ ]:
efficient_df = pd.read_csv("../predictions/efficient_pred.csv")
efficient_df.head()


In [ ]:
efficient_pred = np.argmax(efficient_df, axis=1)
efficient_pred

In [ ]:
efficient_accuracy = accuracy_score(y_test, efficient_pred)
efficient_accuracy

In [ ]:
print(classification_report(y_test,efficient_pred,digits=4))

In [ ]:
weights = [0.40, 0.30, 0.30]

In [ ]:
final_pred = np.zeros((efficient_df.shape[0], efficient_df.shape[1]))

In [ ]:
for i in range(efficient_df.shape[0]):
    final_pred[i] = (weights[0] * inception_df.iloc[i,:] + weights[1] * resnet_df.iloc[i,:] + weights[2] * efficient_df.iloc[i,:]) / sum(weights)

In [ ]:
# for i in range(efficient_df.shape[0]):
#     final_pred[i] = (weights[0] * inception_df.iloc[i,:] + weights[1] * resnet_df.iloc[i,:] ) / sum(weights)

In [ ]:
final_pred

In [ ]:
y_pred = np.argmax(final_pred, axis=1)

In [ ]:
y_pred

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
print(classification_report(y_test,y_pred,digits=4))

In [ ]:
# The class-index -> class-name map comes from the ground-truth file, so this notebook
# does not need the image folder at all.
swapped_dict = dict(zip(gt["actual"], gt["actual_label"]))
filenames = gt["image"].tolist()
print(len(filenames), "images,", len(swapped_dict), "classes")


In [ ]:
len(filenames)


In [ ]:
# (class map built above)


In [ ]:
resnet_labels = [swapped_dict.get(item, item) for item in resnet_pred]

In [ ]:
resnet_labels

In [ ]:
efficient_labels = [swapped_dict.get(item, item) for item in efficient_pred]
efficient_labels

In [ ]:
inception_labels = [swapped_dict.get(item, item) for item in inception_pred]
inception_labels

In [ ]:
test_labels = [swapped_dict.get(item, item) for item in y_test]
test_labels

In [ ]:
voting_labels = [swapped_dict.get(item, item) for item in y_pred]
voting_labels

In [ ]:
voting_df = pd.DataFrame()
voting_df.loc[:,'image'] = filenames
voting_df.loc[:,'resnet_pred'] = resnet_pred
voting_df.loc[:,'resnet_label'] = resnet_labels
voting_df.loc[:,'efficient_pred'] = efficient_pred
voting_df.loc[:,'efficient_label'] = efficient_labels
voting_df.loc[:,'inception_pred'] = inception_pred
voting_df.loc[:,'inception_label'] = inception_labels
voting_df.loc[:,'voting_pred'] = y_pred
voting_df.loc[:,'voting_label'] = voting_labels
voting_df.loc[:,'actual'] = y_test
voting_df.loc[:,'actual_label'] = test_labels


In [ ]:
# Written under a new name so the shipped artifact is never overwritten.
voting_df.to_csv("combined_model_predictions_recomputed.csv", index=False)
print('wrote combined_model_predictions_recomputed.csv')
